In [1]:
import vtk
import matplotlib.pyplot as plt
import numpy as np
import os
from vtk import vtkXMLUnstructuredGridReader
from vtk.util.numpy_support import vtk_to_numpy

import matplotlib.patches as patches
from scipy.spatial import KDTree
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patheffects as pe

import pygmt

In [2]:
base_dir = "/Users/danieldouglas/FINAL_SLABS/plate_model/tar_files/"

solution_dir_path = base_dir + "dry/final_suite/5e20_10kmweak_dry_NEW/solution/"

solutions = np.sort(os.listdir( solution_dir_path ))
t_step = 0

if 'pos_solution' in locals():
    del pos_solution, density_total, crust_total, litho_total, UM_total

for soln in solutions:
    if soln[-4:-1] == '.vt' and soln[9:14] == str(t_step).zfill(5):
        file_path = os.path.join(solution_dir_path, soln)
        reader = vtkXMLUnstructuredGridReader()
        reader.SetFileName(file_path)
        reader.Update()
        data = reader.GetOutput()
        points = data.GetPoints()
        x = vtk_to_numpy(points.GetData())
        crust = vtk_to_numpy(data.GetPointData().GetArray('Crust'))
        litho = vtk_to_numpy(data.GetPointData().GetArray('Base_Subducting'))
        density = vtk_to_numpy(data.GetPointData().GetArray("density"))
        UM = vtk_to_numpy(data.GetPointData().GetArray("Upper_Mantle"))

        if 'pos_solution' in locals():
            pos_solution = np.concatenate( (pos_solution, x) )
            crust_total = np.concatenate( (crust_total, crust) )
            litho_total = np.concatenate( (litho_total, litho) )
            density_total = np.concatenate( (density_total, density) )
            UM_total = np.concatenate( (UM_total, UM) )
        else:
            pos_solution = x
            crust_total = crust
            litho_total = litho
            density_total = density
            UM_total = UM

In [3]:
def cartesian_to_spherical(x, y, z):
    """
    Takes an x, y, z and converts it to spherical coordinates. Returns r, theta, phi
    """
    r = np.sqrt(x**2 + y**2 + z**2)
    theta = 90 - np.rad2deg( np.arccos( z / (np.sqrt(x**2 + y**2 + z**2)) ) )
    phi =  np.sign(y) * np.rad2deg(np.arccos( x / np.sqrt(x**2 + y**2) ))
    phi[np.where(phi < 0)] = phi[np.where(phi < 0)] + 360
    phi[np.where(phi == 0)] = 180
    
    return r, phi, theta

def spherical_to_global_cartesian(r, phi, theta):
    """
    Takes spherical coordinates r, theta, and phi and converts to Cartesian coordinates.
    Returns x, y, z
    """
    x = r * np.sin(np.deg2rad(90 - theta)) * np.cos(np.deg2rad(phi))
    y = r * np.sin(np.deg2rad(90 - theta)) * np.sin(np.deg2rad(phi))
    z = r * np.cos(np.deg2rad(90 - theta))
    
    return x, y, z

In [16]:
ASP_r, ASP_phi, ASP_theta = cartesian_to_spherical(pos_solution[:, 0], pos_solution[:, 1], pos_solution[:, 2])
max_R = 6250e3

litho_inds = np.where( (litho_total >= 0.5) & (ASP_r <= max_R) )
crust_inds = np.where( (crust_total >= 0.5) & (ASP_r <= max_R) )
UM_inds = np.where( (UM_total >= 0.5) & (ASP_r <= max_R) )

litho_dense = np.average(density_total[litho_inds])
crust_dense = np.average(density_total[crust_inds])
UM_dense = np.average(density_total[UM_inds])

crust_thickness = 10e3
litho_thickness = 110e3

trench_length   = 2300e3
slab_length     = 500e3

In [23]:
total_slab_force = (litho_dense - UM_dense) * litho_thickness * trench_length * slab_length + \
                   (crust_dense - UM_dense) * crust_thickness * trench_length * slab_length
print(total_slab_force)

1.881191394042969e+19


In [24]:
total_slab_force = (litho_dense - UM_dense) * (litho_thickness + 4e3) * trench_length * slab_length + \
                   (crust_dense - UM_dense) * (crust_thickness - 4e3) * trench_length * slab_length
print(total_slab_force)

1.8038340161132812e+19


In [25]:
1.88119 / 1.80383

1.0428865247833776

In [21]:
litho_dense

3321.919

In [22]:
UM_dense

3199.6147